In [61]:
import pandas as pd


df_comments=pd.read_csv(f'../../data/df_n4.csv')
df_context=pd.read_csv(f'../../data/df_n2.csv')


In [62]:

import ast

df = df_comments.merge(
    df_context[
        [
            "patch_id",
            "relevant_context",
            "hunk",
            "target_file",
            "pr_title",
            "changed_files",
            "relevant_same_file_code_hunks"
        ]
    ],
    on="patch_id",
    how="left",
)

def get_first_comment(comments):
    """
    After deduplication, every cluster should have one comment representing it:
    duplicate cluster → use the synthesized comment generated by the LLM;
    singleton cluster → there is nothing to synthesize, so just use the original comment.
    """
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except Exception:
            return comments

    if isinstance(comments, list):
        return comments[0] if len(comments) > 0 else None

    return comments

df["comment"] = df.apply(
    lambda row: (
        row["synthesis_comment"]
        if pd.notna(row["synthesis_comment"])
        else get_first_comment(row["comments"])
    ),
    axis=1,
)

df=df.drop(columns=['comments','synthesis_comment'])


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1514 entries, 0 to 1513
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   patch_id                       1514 non-null   object
 1   num_comments                   1514 non-null   int64 
 2   cluster_id                     1514 non-null   object
 3   comment_ids                    1514 non-null   object
 4   generation_systems             1514 non-null   object
 5   categories                     1514 non-null   object
 6   severities                     1514 non-null   object
 7   agreement_scores               1514 non-null   object
 8   relevant_context               1514 non-null   object
 9   hunk                           1514 non-null   object
 10  target_file                    1514 non-null   object
 11  pr_title                       1514 non-null   object
 12  changed_files                  1514 non-null   object
 13  rel

CREATE BATCH JSON

In [32]:
import json
import os
import pandas as pd

MODEL_NAME = "gpt-5.6-luna"
REASONING_EFFORT = "medium"

BATCH_INPUT_PATH = "comment_assessment_batch.jsonl"


# -----------------------------
# PROMPT BUILDING
# -----------------------------
def build_messages(row):

    system_prompt = """
You are a highly skilled software engineer with extensive experience reviewing pull requests.

Your task is to evaluate the relevance of a code review comment for a given code change.

A highly relevant review comment:
- identifies a real issue or meaningful improvement related to the code change,
- is supported by the provided patch and surrounding context,
- is specific and actionable,
- is concise without omitting important information,
- does not discuss unrelated or pre-existing issues.

A low-relevance review comment:
- discusses code unrelated to the patch,
- makes unsupported assumptions,
- is factually incorrect,
- is too vague or generic,
- or provides little useful value.

Evaluate the review comment using ONLY the provided pull request information, code patch, and surrounding context. Do not assume code or project behavior that is not shown.

Assign a relevance score from 1 to 5 using the following rubric:

1 = Completely irrelevant
2 = Mostly irrelevant
3 = Partially relevant
4 = Mostly relevant
5 = Highly relevant

Return ONLY valid JSON in the following format:

{
  "comment_id": "<comment ID>",
  "score": <integer between 1 and 5>,
  "reason": "<short explanation>"
}

The comment_id MUST be exactly the comment ID provided in the input.
Do not modify, truncate, or invent the comment ID.

Do not return any text outside the JSON object.
"""

    user_prompt = f"""
COMMENT ID:
{row['cluster_id']}

PULL REQUEST TITLE:
{row['pr_title']}

TARGET FILE:
{row['target_file']}

RELATED CODE HUNKS IN THE SAME FILE:
<related_hunks>
{row['relevant_same_file_code_hunks']}
</related_hunks>

SURROUNDING CONTEXT:
<context>
{row['relevant_context']}
</context>

CODE PATCH:
<patch>
{row['hunk']}
</patch>

REVIEW COMMENT:
{row['comment']}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def create_batch_jsonl(df, output_path):
    """
    Create an OpenAI Batch API JSONL file.

    One JSONL line = one comment assessment request.
    """

    output_dir = os.path.dirname(output_path)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    num_requests = 0
    skipped = 0

    with open(output_path, "w", encoding="utf-8") as f:

        for idx, row in df.iterrows():

            # Same validation as the original pipeline
            if (
                pd.isna(row["hunk"])
                or row["comment"] is None
                or str(row["comment"]).strip() == ""
            ):
                skipped += 1
                continue

            messages = build_messages(row)

            request = {
                "custom_id": f"comment_assessment_{idx}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL_NAME,
                    "messages": messages,
                }
            }

            if REASONING_EFFORT:
                request["body"]["reasoning_effort"] = REASONING_EFFORT

            f.write(
                json.dumps(request, ensure_ascii=False)
                + "\n"
            )

            num_requests += 1

    print(f"[BATCH] Created JSONL: {output_path}")
    print(f"[BATCH] Requests: {num_requests}")
    print(f"[BATCH] Skipped rows: {skipped}")

    return output_path

In [33]:
create_batch_jsonl(
    df,
    BATCH_INPUT_PATH
)

[BATCH] Created JSONL: comment_assessment_batch.jsonl
[BATCH] Requests: 1514
[BATCH] Skipped rows: 0


'comment_assessment_batch.jsonl'

SUBLIT BATCH

In [34]:

 
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


In [35]:


INPUT_FILE = "comment_assessment_batch.jsonl"


print("[1/2] Uploading JSONL...")

with open(INPUT_FILE, "rb") as f:
    batch_file = client.files.create(
        file=f,
        purpose="batch"
    )

print(f"[UPLOAD] File ID: {batch_file.id}")


print("[2/2] Creating Batch...")

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"[BATCH] Batch ID: {batch.id}")
print(f"[BATCH] Status: {batch.status}")


[1/2] Uploading JSONL...
[UPLOAD] File ID: file-WjWqsNRd81z7iRFhGkQfEo
[2/2] Creating Batch...
[BATCH] Batch ID: batch_6a9a12fb62e08190938081c26712b37d
[BATCH] Status: validating


In [45]:

from openai import OpenAI
from dotenv import load_dotenv



# Use the batch ID returned when you submitted the batch
BATCH_ID = batch.id

current_batch = client.batches.retrieve(BATCH_ID)

print("Batch ID:", current_batch.id)
print("Status:", current_batch.status)

if current_batch.request_counts:
    print("Total requests:", current_batch.request_counts.total)
    print("Completed:", current_batch.request_counts.completed)
    print("Failed:", current_batch.request_counts.failed)

if current_batch.output_file_id:
    print("Output file ID:", current_batch.output_file_id)

if current_batch.error_file_id:
    print("Error file ID:", current_batch.error_file_id)
    
    

Batch ID: batch_6a9a12fb62e08190938081c26712b37d
Status: completed
Total requests: 1514
Completed: 1514
Failed: 0
Output file ID: file-U97U8No5b4reYk6ebqU812


VERIFY STATE OF BATCH SUBMISSION

GET BATCH SUBMISSION RESULTS

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os

load_dotenv()

client = OpenAI()

# Use the batch ID returned when you submitted the batch
BATCH_ID = batch.id

# Retrieve batch information
current_batch = client.batches.retrieve(BATCH_ID)

print("Batch ID:", current_batch.id)
print("Status:", current_batch.status)

if current_batch.request_counts:
    print("Total requests:", current_batch.request_counts.total)
    print("Completed:", current_batch.request_counts.completed)
    print("Failed:", current_batch.request_counts.failed)

# ---------------------------------------------------------
# Retrieve successful results
# ---------------------------------------------------------

if current_batch.output_file_id:

    print("Output file ID:", current_batch.output_file_id)

    # Download the output file
    output_file = client.files.content(current_batch.output_file_id)

    # Decode the file content
    content = output_file.text

    # Batch output is JSONL: one JSON object per line
    results = [
        json.loads(line)
        for line in content.splitlines()
        if line.strip()
    ]

    # Save as a regular JSON file
    output_path = "batch_results.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Results saved to: {output_path}")
    print(f"Number of results: {len(results)}")


# ---------------------------------------------------------
# Retrieve failed requests, if any
# ---------------------------------------------------------

if current_batch.error_file_id:

    print("Error file ID:", current_batch.error_file_id)

    error_file = client.files.content(current_batch.error_file_id)

    error_content = error_file.text

    errors = [
        json.loads(line)
        for line in error_content.splitlines()
        if line.strip()
    ]

    error_path = "batch_errors.json"

    with open(error_path, "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=2, ensure_ascii=False)

    print(f"Errors saved to: {error_path}")
    print(f"Number of errors: {len(errors)}")

GET JSON RESULTS and transform them to df_n4

In [ ]:
import pandas as pd

jsonl_path = "comment_assessment_batch_results.jsonl"

df = pd.read_json(jsonl_path, lines=True)

In [54]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [55]:
df['response'].head()

0                    {'status_code': 200, 'request_id': '8a15df77-ccd8-4674-845e-b2d1bebb6df5', 'body': {'id': 'chatcmpl-EKCG56d7NmAk6NCk2NvDr5zUwJ1QW', 'object': 'chat.completion', 'created': 1788482389, 'model': 'gpt-5.6-luna', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{"comment_id":"P000000_c0","score":5,"reason":"The comment directly addresses the newly added docstring and correctly notes that the handler always returns 204 without checking scheduler state, which is important for accurately describing the endpoint’s behavior."}', 'refusal': None, 'annotations': []}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 618, 'completion_tokens': 133, 'total_tokens': 751, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 72, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': None}}
1

In [56]:

def parse_batch_result(text):

    if text is None:
        return None, None

    text = str(text).strip()

    try:
        data = json.loads(text)

        score = data.get("score")

        try:
            score = int(score)
        except Exception:
            score = None

        if score not in [1, 2, 3, 4, 5]:
            score = None

        reason = str(
            data.get("reason", "")
        ).strip()

        return score, reason

    except Exception:
        return None, text




def load_batch_predictions(results_path):

    predictions = {}

    with open(results_path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            result = json.loads(line)

            custom_id = result.get("custom_id")

            if custom_id is None:
                continue

            try:
                response = result["response"]

                body = response["body"]

                choices = body["choices"]

                if not choices:
                    predictions[custom_id] = {
                        "relevance_score": None,
                        "reason": "EMPTY_RESPONSE"
                    }
                    continue

                raw_text = choices[0]["message"]["content"]

                score, reason = parse_batch_result(raw_text)

                predictions[custom_id] = {
                    "relevance_score": score,
                    "reason": reason
                }

            except Exception as e:

                predictions[custom_id] = {
                    "relevance_score": None,
                    "reason": f"BATCH_PARSE_ERROR: {e}"
                }

    print(
        f"[BATCH] Loaded {len(predictions)} predictions"
    )

    return predictions

In [57]:
RESULTS_PATH='comment_assessment_batch_results.jsonl'
predictions = load_batch_predictions(
    RESULTS_PATH
)

[BATCH] Loaded 1514 predictions


In [59]:
print(predictions)

{'comment_assessment_0': {'relevance_score': 5, 'reason': 'The comment directly addresses the newly added docstring and correctly notes that the handler always returns 204 without checking scheduler state, which is important for accurately describing the endpoint’s behavior.'}, 'comment_assessment_1': {'relevance_score': 5, 'reason': 'The comment identifies a real memory leak: the string allocated for the vc3-builder command is not freed after system(cmd). It is directly related to the patch and gives a specific fix.'}, 'comment_assessment_2': {'relevance_score': 5, 'reason': 'The comment identifies a real double-free: cmd is freed inside both worker-command branches and then freed again unconditionally before the runos_os block. Removing one of the frees is specific and actionable.'}, 'comment_assessment_3': {'relevance_score': 1, 'reason': 'The comment is factually incorrect: `cmd` is freed after `system(cmd)` completes, and the shown patch adds `free(cmd);` before the `runos_os` blo

In [64]:
def build_final_dataframe(df, predictions):

    df = df.copy()

    relevance_scores = []
    reasons = []

    for idx, row in df.iterrows():

        custom_id = f"comment_assessment_{idx}"

        # Missing input was never submitted
        if (
            pd.isna(row["hunk"])
            or row["comment"] is None
            or str(row["comment"]).strip() == ""
        ):

            relevance_scores.append(None)
            reasons.append("missing input")

            continue

        # Find batch result
        prediction = predictions.get(custom_id)

        if prediction is None:

            relevance_scores.append(None)
            reasons.append("MISSING_BATCH_RESULT")

        else:

            relevance_scores.append(
                prediction["relevance_score"]
            )

            reasons.append(
                prediction["reason"]
            )

    df["relevance_score"] = relevance_scores
    df["reason"] = reasons

    columns = [
        "patch_id",
        "cluster_id",
        "num_comments",
        "generation_systems",
        "categories",
        "severities",
        "comment",
        "relevance_score",
        "reason",
    ]

    final_df = df[columns].copy()

    print("[DONE] Finished batch post-processing")
    print("Final dataframe shape:", final_df.shape)

    return final_df

In [65]:
final_df = build_final_dataframe(
    df,
    predictions
)


[DONE] Finished batch post-processing
Final dataframe shape: (1514, 9)


In [66]:

final_df.head()

,patch_id,cluster_id,num_comments,generation_systems,categories,severities,comment,relevance_score,reason
0,P000000,P000000_c0,2,"['gpt-5.4-mini-2026-03-17', 'devstral-small-2-24b-instruct-2512']","['Documentation', 'Documentation']","['Low', 'Low']",Ensure the docstring accurately conveys that this health-check endpoint only returns a static 204 and does not validate scheduler state.,5,"The comment directly addresses the newly added docstring and correctly notes that the handler always returns 204 without checking scheduler state, which is important for accurately describing the endpoint’s behavior."
1,P000001,P000001_c0,4,"['qwen3.6-35b-a3b', 'gpt-5.6-sol', 'devstral-small-2-24b-instruct-2512', 'gemma-4-26b-a4b-it']","['Correctness', 'Correctness', 'Correctness', 'Correctness']","['Medium', 'Low', 'High', 'Medium']","The command string allocated for the vc3-builder copy is not freed after system(cmd) completes, causing a leak. Free it on the normal path and before exiting on failure if appropriate.",5,The comment identifies a real memory leak: the string allocated for the vc3-builder command is not freed after system(cmd). It is directly related to the patch and gives a specific fix.
2,P000001,P000001_c1,2,"['gpt-5.4-mini-2026-03-17', 'gpt-5.6-sol']","['Correctness', 'Correctness']","['High', 'High']","This unconditional free duplicates the branch-local frees in the preceding worker-command block, causing a double free. Keep only one free for that allocation.",5,The comment identifies a real double-free: cmd is freed inside both worker-command branches and then freed again unconditionally before the runos_os block. Removing one of the frees is specific and actionable.
3,P000001,P000001_c2,1,['gpt-4o-mini-2024-07-18'],['Correctness'],['High'],"There's a 'free(cmd);' statement missing after the 'if(worker_command != NULL)' block and before it goes to the next block — if 'cmd' is freed after it is used in 'system(cmd)', does this not create a potential use-after-free issue?",1,"The comment is factually incorrect: `cmd` is freed after `system(cmd)` completes, and the shown patch adds `free(cmd);` before the `runos_os` block. No use-after-free is present there."
4,P000001,P000001_c3,1,['devstral-small-2-24b-instruct-2512'],['Correctness'],['Medium'],The `system(cmd)` call in the `runos_os` block doesn't check its return value for consistency with the rest of the function (where all `system` calls are followed by error handling).,1,"The comment is incorrect: the runos_os block stores the result of system(cmd) in k and checks it with if (k), including error handling."


In [67]:
final_df.to_csv('../../data/df_n5.csv', index=False)